# HealthLens v3: TabPFN-3.5 benchmark
Runs TabPFN-3.5 on the **same train/validation/test split** as the local XGBoost and logistic-regression models, and writes `artifacts/model_card.json`, which the server reads to fill the comparison table and screening thresholds.

**Before you start (one time)**
1. Create a free Hugging Face account and accept the licence on https://huggingface.co/Prior-Labs/tabpfn_3_5 (the weights are gated).
2. Create a read token at https://huggingface.co/settings/tokens.
3. In Colab: left sidebar, key icon (Secrets), add `HF_TOKEN` with that token, switch on notebook access.
4. Runtime, Change runtime type, **T4 GPU** (CPU also works but is slow).
5. Zip the `healthlens-v3` folder as `healthlens-v3.zip` and upload it in the next cell.

In [ ]:
!pip -q install "tabpfn>=10.2" xgboost
import os, sys, zipfile, glob
from google.colab import files, userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not os.path.exists("healthlens-v3"):
    up = files.upload()                       # choose healthlens-v3.zip
    zipfile.ZipFile(list(up)[0]).extractall(".")
root = glob.glob("**/backend/risk_model.py", recursive=True)[0].split("/backend")[0]
os.chdir(root); sys.path.insert(0, os.getcwd()); print("project root:", os.getcwd())

In [ ]:
import json, time, numpy as np, pandas as pd, torch
from backend import risk_model as rm
from backend.risk_model import split_data, encode, load_dataset, comparison_row, screening_threshold, FEATURES, CATEGORICAL_IDX
from sklearn.metrics import roc_auc_score
df, info = load_dataset()
train, val, test = split_data(df)
Xtr, Xva, Xte = encode(train), encode(val), encode(test)
ytr, yva, yte = (d["label"].to_numpy(int) for d in (train, val, test))
print(len(train), len(val), len(test), "| GPU:", torch.cuda.is_available())
ctx = pd.read_csv("artifacts/tabpfn_context.csv")
assert ctx[FEATURES].shape[0] == 5000, "context file should have 5,000 rows"
# the context must come from the training split only (no leakage)
key = lambda X: set(map(tuple, np.round(X, 4)))
Cx = ctx[FEATURES].to_numpy(float)          # the context file is already numerically encoded
print("context rows found in train:", len(key(Cx) & key(Xtr)), "of", len(Cx))
print("context rows also in val/test:", len(key(Cx) & (key(Xva) | key(Xte))), "(a handful of identical feature profiles can coincide; that is not leakage of rows)")

In [ ]:
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
clf = TabPFNClassifier.create_default_for_version(ModelVersion.V3_5, n_estimators=1, fit_mode="fit_with_cache",
                                                  categorical_features_indices=CATEGORICAL_IDX)
t0 = time.time(); clf.fit(ctx[FEATURES].to_numpy(float), ctx["label"].to_numpy(int)); t_fit = time.time() - t0

def predict(X, bs=2000):
    return np.concatenate([clf.predict_proba(X[i:i+bs])[:, 1] for i in range(0, len(X), bs)])

t0 = time.time(); pva = predict(Xva); t_val = time.time() - t0
t0 = time.time(); pte = predict(Xte); t_te = time.time() - t0
print(f"fit {t_fit:.1f}s | val {len(Xva)} rows {t_val:.1f}s | test {len(Xte)} rows {t_te:.1f}s")
print("TabPFN-3.5 test AUC:", round(roc_auc_score(yte, pte), 4))

In [ ]:
row = comparison_row("TabPFN-3.5", "primary", yva, pva, yte, pte, extra={"context_rows": len(ctx), "n_estimators": 1})
row.pop("name"); row.pop("role")
thr = {"moderate": row["screening_threshold"], "high": 0.5}
print(row, thr)

# speed of the things the app does per patient (these decide whether the live server is usable on CPU)
one = Xte[:1]
t0 = time.time(); clf.predict_proba(one); t_one = time.time() - t0
ref = Xtr[ytr == 0][:4]
coal = np.tile(Xte[:1], (256 * len(ref), 1))          # exact Shapley = 2^8 coalitions x reference rows
t0 = time.time(); clf.predict_proba(coal); t_shap = time.time() - t0
print(f"single prediction {t_one:.2f}s | Shapley batch ({len(coal)} rows) {t_shap:.1f}s")

In [ ]:
# Compare with the local models on the identical split (reads artifacts written by `python -m backend.risk_model`)
stats = json.load(open("artifacts/stats.json"))
for r in stats.get("comparison", []):
    print(f'{r["name"]:28s} AUC {r.get("test_auc")}  CI {r.get("test_auc_ci95")}  recall {r.get("recall")}  precision {r.get("precision")}')
print(f'{"TabPFN-3.5":28s} AUC {row["test_auc"]}  CI {row["test_auc_ci95"]}  recall {row["recall"]}  precision {row["precision"]}')

In [ ]:
card = {"tabpfn": {**row, "timings_s": {"fit": round(t_fit, 1), "val_predict": round(t_val, 1), "test_predict": round(t_te, 1),
                                         "single_predict": round(t_one, 2), "shapley_batch_1024_rows": round(t_shap, 1)},
                   "device": "cuda" if torch.cuda.is_available() else "cpu"},
        "thresholds": thr, "tabpfn_package": __import__("tabpfn").__version__ if hasattr(__import__("tabpfn"), "__version__") else "unknown"}
json.dump(card, open("artifacts/model_card.json", "w"), indent=2)
files.download("artifacts/model_card.json")
print(json.dumps(card, indent=2))

## After the run
Put the downloaded `model_card.json` into `healthlens-v3/artifacts/` (replace if present), commit, and redeploy the backend. The *About the data and the models* table on the result page then shows the real TabPFN numbers, and the server uses the thresholds measured with TabPFN.

If the TabPFN AUC is not at least as high as XGBoost, report that honestly: on this dataset the label is almost fully determined by HbA1c and glucose, so all models score about the same.